# Annotation analysis

Sentence-level review analysis for geothermal relevance, frame detection, sentiment, and location extraction.

In [5]:
import glob
import json
import os
import sqlite3
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
ANNOTATION_DIR = ROOT / "dutch"/ "databases"
DB_GLOB = str(ANNOTATION_DIR / "**" / "*.db")
all_db_paths = sorted(glob.glob(DB_GLOB, recursive=True))

def has_required_tables(db_path: str, required=("tasks", "annotations")) -> bool:
    con = sqlite3.connect(db_path)
    try:
        tables = pd.read_sql_query(
            "SELECT name FROM sqlite_master WHERE type='table'",
            con,
        )["name"].tolist()
    finally:
        con.close()
    return set(required).issubset(tables)

db_paths = [p for p in all_db_paths if has_required_tables(p)]
if not db_paths:
    raise FileNotFoundError(f"No valid annotation databases found under {ANNOTATION_DIR}")

print("Databases:")
for p in db_paths:
    print("-", p)


Databases:
- /Users/Yannick/dev/geothermal_text_mining/annotation/dutch/databases/annotations.db


In [6]:
def load_annotation_db(db_path: str) -> pd.DataFrame:
    con = sqlite3.connect(db_path)
    try:
        tasks = pd.read_sql_query("SELECT * FROM tasks", con)
        anns = pd.read_sql_query("SELECT * FROM annotations", con)
    finally:
        con.close()

    df = anns.merge(tasks, on="paragraph_uid", how="left", suffixes=("_ann", "_task"))
    return df

df = pd.concat([load_annotation_db(p) for p in db_paths], ignore_index=True)
print(df.shape)
df.head()


(13, 18)


,paragraph_uid,annotator,geothermal_relevant,sentiment_correct,sentiment_true,matched_categories_present,matched_categories_correct,matched_categories_true,location_correct,location_true,keywords_to_add,notes,created_at,paragraph_text,aspect_pred,sentiment_pred,split_bucket,meta_json
0,04683a743427ad09221010fb0fd1570fbd672b0867bb66...,Egberink,1,1,None,1,1,None,1,None,None,None,2026-04-09T10:43:48.660582,En als dat college zo graag een verbod op gasw...,Governance,negative,7,"{""source_file"": ""Bestanden (501-1000) (1).RTF""..."
1,05f19a71d26d78a2331f17cf51b3ecef1f65876d8891b4...,Egberink,1,1,None,1,1,None,1,None,None,None,2026-04-09T10:44:00.801708,Direct na het incident startte Staatstoezicht ...,Risks,negative,15,"{""source_file"": ""1-500.RTF"", ""source_path"": ""/..."
2,08def3e22fb14e01cb2084be1d273413a8cc859a79fcd1...,Egberink,1,1,None,1,1,None,1,None,None,None,2026-04-09T10:44:22.759512,De initiatiefnemers beloven dat hun warmte nie...,Costs,positive,17,"{""source_file"": ""Bestanden (500) (5).RTF"", ""so..."
3,0a876f224feaf34350abc5710c93bebb470781c07ba599...,Egberink,1,1,None,1,1,None,1,None,None,None,2026-04-09T10:44:55.221245,Vergunning Lastig In juni vorig jaar besliste ...,Governance,negative,13,"{""source_file"": ""Bestanden (500) (5).RTF"", ""so..."
4,0c7291ba2535dce335797e94228fac3e60398681878acd...,Egberink,1,1,None,1,1,None,1,None,None,None,2026-04-09T10:47:27.201349,Volgens de Rekenkamer ontbreekt regie van de o...,Governance,negative,17,"{""source_file"": ""Bestanden (1500-1862).RTF"", ""..."


In [7]:
def parse_listish(value):
    if value is None or pd.isna(value):
        return []
    s = str(value).strip()
    if not s or s.lower() in {"nan", "none", "null"}:
        return []
    return [part.strip() for part in s.split(";") if part.strip()]

def parse_meta(value):
    if value is None or pd.isna(value):
        return {}
    if isinstance(value, dict):
        return value
    try:
        return json.loads(value)
    except Exception:
        return {}

analysis_df = df.copy()
analysis_df["meta"] = analysis_df["meta_json"].map(parse_meta)
analysis_df["sentence_uid"] = analysis_df["paragraph_uid"]
analysis_df["sentence_text"] = analysis_df["meta"].map(lambda m: m.get("sentence_text") or m.get("paragraph_text") or "")
analysis_df["paragraph_text_context"] = analysis_df["meta"].map(lambda m: m.get("paragraph_text") or "")
analysis_df["predicted_frames"] = analysis_df["meta"].map(lambda m: parse_listish(m.get("matched_categories_str") or m.get("aspect_pred")))
analysis_df["predicted_location"] = analysis_df["meta"].map(lambda m: m.get("predicted_location") or m.get("llm_location"))
analysis_df["matched_location"] = analysis_df["meta"].map(lambda m: m.get("matched_location") or m.get("geo_name_matched"))
analysis_df["predicted_sentiment"] = analysis_df["sentiment_pred"]
analysis_df["true_frames"] = analysis_df["matched_categories_true"].map(parse_listish)
analysis_df["geothermal_relevant"] = analysis_df["geothermal_relevant"].map({1: True, 0: False})
analysis_df["sentiment_correct"] = analysis_df["sentiment_correct"].map({1: True, 0: False})
analysis_df["matched_categories_correct"] = analysis_df["matched_categories_correct"].map({1: True, 0: False})
analysis_df["location_correct"] = analysis_df["location_correct"].map({1: True, 0: False})
analysis_df[["annotator", "sentence_uid", "sentence_text", "geothermal_relevant", "predicted_sentiment", "predicted_frames", "predicted_location", "matched_location"]].head()


,annotator,sentence_uid,sentence_text,geothermal_relevant,predicted_sentiment,predicted_frames,predicted_location,matched_location
0,Egberink,04683a743427ad09221010fb0fd1570fbd672b0867bb66...,En als dat college zo graag een verbod op gasw...,True,negative,[Governance],Pernis,Rotterdam
1,Egberink,05f19a71d26d78a2331f17cf51b3ecef1f65876d8891b4...,Direct na het incident startte Staatstoezicht ...,True,negative,[Risks],Maasdijk,Westland
2,Egberink,08def3e22fb14e01cb2084be1d273413a8cc859a79fcd1...,De initiatiefnemers beloven dat hun warmte nie...,True,positive,[Costs],Noordwijk,Noordwijk
3,Egberink,0a876f224feaf34350abc5710c93bebb470781c07ba599...,Vergunning Lastig In juni vorig jaar besliste ...,True,negative,[Governance],Horst,Horst aan de Maas
4,Egberink,0c7291ba2535dce335797e94228fac3e60398681878acd...,Volgens de Rekenkamer ontbreekt regie van de o...,True,negative,[Governance],Katwijk,Katwijk


In [8]:
summary_metrics = {
    "n_annotations": len(analysis_df),
    "n_unique_sentences": analysis_df["sentence_uid"].nunique(),
    "n_annotators": analysis_df["annotator"].nunique(),
}

for col in ["geothermal_relevant", "sentiment_correct", "matched_categories_correct", "location_correct"]:
    if col in analysis_df.columns and analysis_df[col].notna().any():
        summary_metrics[f"{col}_rate"] = analysis_df[col].mean()

pd.Series(summary_metrics).to_frame("value")


,value
n_annotations,13.000000
n_unique_sentences,13.000000
n_annotators,1.000000
geothermal_relevant_rate,1.000000
sentiment_correct_rate,0.692308
matched_categories_correct_rate,1.000000
location_correct_rate,1.000000


In [9]:
review_breakdown = pd.DataFrame({
    "geothermal_relevant": analysis_df["geothermal_relevant"].value_counts(dropna=False),
    "sentiment_correct": analysis_df["sentiment_correct"].value_counts(dropna=False),
    "frame_correct": analysis_df["matched_categories_correct"].value_counts(dropna=False),
    "location_correct": analysis_df["location_correct"].value_counts(dropna=False),
}).fillna(0).astype(int)
review_breakdown


,geothermal_relevant,sentiment_correct,frame_correct,location_correct
False,0,4,0,0
True,13,9,13,13


In [10]:
non_geothermal = analysis_df[analysis_df["geothermal_relevant"] == False].copy()
non_geothermal[["sentence_uid", "sentence_text", "predicted_frames", "predicted_sentiment", "predicted_location", "notes"]].head(20)


,sentence_uid,sentence_text,predicted_frames,predicted_sentiment,predicted_location,notes


In [11]:
overlap = analysis_df.groupby(["sentence_uid", "annotator"]).size().reset_index(name="n")
paired = analysis_df.groupby("sentence_uid").filter(lambda g: g["annotator"].nunique() > 1).copy()
print("Overlap sentences:", paired["sentence_uid"].nunique())

def agreement_rate(frame):
    if frame.empty:
        return None
    return frame.groupby("sentence_uid").apply(lambda g: g.nunique() == 1).mean()

agreement = {
    "geothermal_relevant": agreement_rate(paired[["sentence_uid", "geothermal_relevant"]].dropna()),
    "sentiment_correct": agreement_rate(paired[["sentence_uid", "sentiment_correct"]].dropna()),
    "matched_categories_correct": agreement_rate(paired[["sentence_uid", "matched_categories_correct"]].dropna()),
    "location_correct": agreement_rate(paired[["sentence_uid", "location_correct"]].dropna()),
}
pd.Series(agreement).to_frame("agreement_rate")


Overlap sentences: 0


,agreement_rate
geothermal_relevant,None
sentiment_correct,None
matched_categories_correct,None
location_correct,None


In [12]:
frame_corrections = analysis_df.loc[analysis_df["matched_categories_correct"] == False, [
    "sentence_uid", "sentence_text", "predicted_frames", "true_frames", "keywords_to_add", "notes"
]].copy()
frame_corrections.head(20)


,sentence_uid,sentence_text,predicted_frames,true_frames,keywords_to_add,notes


In [13]:
location_corrections = analysis_df.loc[analysis_df["location_correct"] == False, [
    "sentence_uid", "sentence_text", "predicted_location", "matched_location", "location_true", "notes"
]].copy()
location_corrections.head(20)


,sentence_uid,sentence_text,predicted_location,matched_location,location_true,notes
